# Sprint 33 — Synthèse énergie & coût matériel

Relie les **métriques de coût matériel-agnostiques** (MACs / FLOPs / BOPs / Params,
`compute_cost.py` ; temps-HW / FLOPS-W, `hw_cost_model.py`) aux **mesures énergie réelles**
(µJ/phase LPM01A, `experiments/exp_S33_energy/`) et à l'**autonomie** (`autonomy.py`).

**Question Gap 3** (CR 9 juin 2026) : l'INT8 réduit-il l'énergie **même sans accélérer la
latence FPU** sur Cortex-M4 (constat Sprint 29) ?

> ⚠️ Règle « aucun chiffre inventé » : les champs énergie/autonomie valent « à mesurer »
> tant que le PowerShield LPM01A n'a pas réellement tourné. Les métriques de coût
> (MACs/FLOPs/BOPs) et le temps-HW proxy sont, eux, calculés réellement.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.evaluation.compute_cost import (
    compute_macs, compute_flops_for_model, compute_bops_for_model,
    compute_params_for_model,
)
from src.evaluation import hw_cost_model as hw
from src.evaluation import autonomy as auto

A_MESURER = auto.A_MESURER
ENERGY_DIR = ROOT / "experiments" / "exp_S33_energy"
HW_PROFILE = ROOT / "configs" / "hw_profile_f439zi.yaml"
PHASES = ("startup", "acquisition", "inference", "idle")
print("Racine projet :", ROOT)

Racine projet : /home/leonard/Documents/ENAC/cl-embedded


## 1. Chargement — JSON énergie (S3306) + profil HW

In [2]:
profile = hw.load_hw_profile(HW_PROFILE)["hardware"]
flops_peak = {"fp32": profile["flops_peak_fp32"], "int8": profile["flops_peak_int8"]}
eff = profile["efficacite"]
tension_v = profile["puissance_watts"]["tension_v"]
actif_mA = profile["puissance_watts"]["actif_mA"]  # None tant que LPM01A non mesuré

energy = {}
for jp in sorted(ENERGY_DIR.glob("*_*.json")):
    if jp.stem in ("autonomy",):
        continue
    d = json.loads(jp.read_text(encoding="utf-8"))
    energy[jp.stem] = d
print("JSON énergie chargés :", list(energy))
print("Courant actif (profil HW) :", actif_mA, "→ FLOPS/W et énergie =", A_MESURER if actif_mA is None else "mesuré")

JSON énergie chargés : ['ewc_fp32', 'ewc_int8', 'hdc_fp32', 'hdc_int8', 'maha_fp32', 'maha_int8', 'tinyol_fp32', 'tinyol_int8']
Courant actif (profil HW) : None → FLOPS/W et énergie = à mesurer


## 2. µJ par phase — barres groupées FP32 vs INT8

Affiché proprement comme « à mesurer » tant que la campagne LPM01A n'a pas tourné.

In [3]:
def phase_value(stem, phase):
    v = energy.get(stem, {}).get("phases_uj", {}).get(phase, A_MESURER)
    return v if isinstance(v, (int, float)) else np.nan

models = ["ewc", "hdc", "tinyol", "maha"]
measured_energy = not all(
    np.isnan(phase_value(f"{m}_{e}", p))
    for m in models for e in ("fp32", "int8") for p in PHASES
)

if measured_energy:
    x = np.arange(len(PHASES)); w = 0.1
    fig, ax = plt.subplots(figsize=(9, 4))
    for i, (m, e) in enumerate([(m, e) for m in models for e in ("fp32", "int8")]):
        ax.bar(x + i * w, [phase_value(f"{m}_{e}", p) for p in PHASES], w, label=f"{m}_{e}")
    ax.set_xticks(x + w * 4); ax.set_xticklabels(PHASES); ax.set_ylabel("µJ"); ax.legend(ncol=4, fontsize=7)
    ax.set_title("Énergie par phase — FP32 vs INT8"); plt.tight_layout(); plt.show()
else:
    print(f"µJ par phase : {A_MESURER} (campagne LPM01A non encore exécutée — S3306).")
    print("Structure prête : 4 modèles ×", PHASES)

µJ par phase : à mesurer (campagne LPM01A non encore exécutée — S3306).
Structure prête : 4 modèles × ('startup', 'acquisition', 'inference', 'idle')


## 3. Table FP32 vs INT8 — énergie (delta_uj / ratio)

In [4]:
summary_p = ENERGY_DIR / "summary.json"
summary = json.loads(summary_p.read_text(encoding="utf-8")) if summary_p.is_file() else {}
rows = []
for m in models:
    e = summary.get("per_model", {}).get(m, {})
    rows.append({"model": m, "fp32_uj": e.get("fp32", A_MESURER),
                 "int8_uj": e.get("int8", A_MESURER),
                 "delta_uj": e.get("delta_uj", A_MESURER),
                 "ratio_int8_fp32": e.get("ratio_int8_fp32", A_MESURER)})
df_energy = pd.DataFrame(rows)
print(summary.get("gap3_note", ""))
df_energy

Sprint 29 : l'INT8 réduit la RAM sans accélérer la latence (FPU Cortex-M4, pas de NPU INT8). Question énergie : l'INT8 réduit-il néanmoins les µJ (moins d'accès mémoire) ? — réponse via mesures LPM01A réelles, champs 'à mesurer' tant que non capturées.


,model,fp32_uj,int8_uj,delta_uj,ratio_int8_fp32
0,ewc,à mesurer,à mesurer,à mesurer,à mesurer
1,hdc,à mesurer,à mesurer,à mesurer,à mesurer
2,tinyol,à mesurer,à mesurer,à mesurer,à mesurer
3,maha,à mesurer,à mesurer,à mesurer,à mesurer


## 4. Pareto énergie / AUROC

In [5]:
pareto = []
for m in models:
    for e in ("fp32", "int8"):
        tot = energy.get(f"{m}_{e}", {}).get("total_uj", A_MESURER)
        pareto.append((f"{m}_{e}", tot))
if any(isinstance(t, (int, float)) for _, t in pareto):
    pass  # tracer scatter (µJ, AUROC) — AUROC issu des exp Sprint 28 quand énergie mesurée
else:
    print(f"Pareto énergie/AUROC : {A_MESURER} (énergie totale non mesurée).")
    print("À tracer dès que total_uj réel + AUROC (exp_S28_*) disponibles.")

Pareto énergie/AUROC : à mesurer (énergie totale non mesurée).
À tracer dès que total_uj réel + AUROC (exp_S28_*) disponibles.


## 5. Coût matériel-agnostique — MACs / FLOPs / BOPs / temps-HW (calculé réellement)

Dimensions board (configs `board_*`) : EWC 5→32→16→2 · HDC D=1000, 5 feat, 2 cl ·
TinyOL_AE 5→32→16 · Mahalanobis 5 feat.

In [6]:
SPEC = {
    "ewc":  ("EWC",        dict(n_features=5, hidden_dims=[32, 16], n_classes=2)),
    "hdc":  ("HDC",        dict(n_features=5, dim_hv=1000, n_classes=2)),
    "tinyol": ("TinyOL_AE", dict(n_features=5, encoder_dims=[32, 16])),
    "maha": ("Mahalanobis", dict(n_features=5)),
}
rows = []
for name, (key, kw) in SPEC.items():
    macs = compute_macs(key, **kw)
    flops = compute_flops_for_model(key, **kw)
    bops_fp32 = compute_bops_for_model(key, 32, **kw)
    bops_int8 = compute_bops_for_model(key, 8, **kw)
    params = compute_params_for_model(key, **kw)
    t_fp32 = hw.estimate_inference_time(macs, flops_peak["fp32"], eff["fp32"])
    rows.append({
        "model": name, "MACs": macs, "FLOPs": flops, "params": params,
        "BOPs_fp32": bops_fp32, "BOPs_int8": bops_int8,
        "BOPs_ratio": bops_fp32 / bops_int8,
        "t_HW_us (proxy)": t_fp32 * 1e6,
        "throughput_inf_s": hw.throughput(t_fp32),
    })
df_cost = pd.DataFrame(rows)
print("BOPs_ratio attendu = (32/8)² = 16 →", set(round(r, 6) for r in df_cost.BOPs_ratio))
df_cost

BOPs_ratio attendu = (32/8)² = 16 → {16.0}


,model,MACs,FLOPs,params,BOPs_fp32,BOPs_int8,BOPs_ratio,t_HW_us (proxy),throughput_inf_s
0,ewc,704,1408,754,1441792,90112,16.0,26.074074,38352.272727
1,hdc,7000,14000,7000,14336000,896000,16.0,259.259259,3857.142857
2,tinyol,1349,2698,1429,2762752,172672,16.0,49.962963,20014.825797
3,maha,30,60,30,61440,3840,16.0,1.111111,900000.000000


## 6. FLOPS/W & autonomie vs capacité batterie

FLOPS/W exige la **puissance mesurée** (LPM01A) → « à mesurer ». Le balayage de capacités
montre la **structure réelle** ; l'autonomie chiffrée arrive avec I_moy mesuré.

In [7]:
caps = auto.load_battery_capacities(HW_PROFILE)
if actif_mA is not None:
    pw = hw.power_watts(actif_mA, tension_v)
    print("FLOPS/W (fp32) :", hw.flops_per_watt(flops_peak["fp32"], pw))
    sweep = auto.sweep_capacities(actif_mA, caps)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(list(sweep), list(sweep.values()), "o-")
    ax.set_xlabel("Capacité (mAh)"); ax.set_ylabel("Autonomie (h)")
    ax.set_title("Autonomie vs capacité batterie"); plt.tight_layout(); plt.show()
else:
    print(f"FLOPS/W : {A_MESURER} (puissance LPM01A non mesurée).")
    print(f"Autonomie : {A_MESURER}. Capacités batterie balayées (config) :", caps)
    auto_p = ENERGY_DIR / "autonomy.json"
    if auto_p.is_file():
        print("autonomy.json :", json.loads(auto_p.read_text())['per_model'].get('ewc_fp32'))

FLOPS/W : à mesurer (puissance LPM01A non mesurée).
Autonomie : à mesurer. Capacités batterie balayées (config) : [220.0, 1200.0, 2000.0, 3000.0, 10000.0]
autonomy.json : {'i_moy_ma': 'à mesurer', 'autonomy_h_by_mah': {'220.0': 'à mesurer', '1200.0': 'à mesurer', '2000.0': 'à mesurer', '3000.0': 'à mesurer', '10000.0': 'à mesurer'}}


## 7. Synthèse — l'INT8 réduit-il l'énergie sans accélérer la latence FPU ?

**Acquis (calculé réellement) :**
- La comparaison **BOPs** rend le gain INT8 quantitatif et honnête : à FLOPs égaux,
  `BOPs_fp32 / BOPs_int8 = (32/8)² = 16` (cf. §5). C'est l'argument central du CR 19 mai.
- Côté **débit/temps-HW proxy**, `flops_peak_int8 == flops_peak_fp32` dans le profil
  (pas de NPU/accélérateur INT8 sur Cortex-M4) → **aucune accélération latence attendue**,
  cohérent avec le constat Sprint 29.

**En attente de mesure LPM01A (« à mesurer ») :**
- µJ/phase, table énergie FP32 vs INT8 (§2–3), Pareto énergie/AUROC (§4), FLOPS/W et
  autonomie chiffrée (§6).

**Conclusion différée** : l'hypothèse — l'INT8 réduit l'énergie (moins d'accès mémoire)
**malgré** l'absence d'accélération latence — ne sera tranchée qu'avec les µJ réels du
PowerShield. Aucune conclusion n'est fabriquée avant exécution (règle CLAUDE.md).